# ComfyUI on Colab → 外部公開セットアップ

このノートブックは、Google Colab上にComfyUIをセットアップし、`cloudflared`で外部からアクセス可能な公開URLを発行します。
発行されたURLを、手元の `app/index.html`（Comfy Simple Studio）の「バックエンドURL」欄に貼り付ければ使えます。

**手順**
1. ランタイムタイプを GPU に変更（ランタイム → ランタイムのタイプを変更 → T4以上）
2. 上から順に全セルを実行
3. 最後のセルに表示される `https://xxxx.trycloudflare.com` をコピー
4. Colabは一定時間操作がないと切断されます。切断されたら最後のセルを再実行（URLは変わります）してください。

In [1]:
# 1. ComfyUI を取得して依存ライブラリをインストール
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -r requirements.txt -q

Cloning into 'ComfyUI'...
remote: Enumerating objects: 43510, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 43510 (delta 19), reused 10 (delta 10), pack-reused 43482 (from 3)
Receiving objects: 100% (43510/43510), 84.26 MiB | 26.21 MiB/s, done.
Resolving deltas: 100% (29527/29527), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 134.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 122.2 MB/s 

In [2]:
# 2. ベースチェックポイントモデルをダウンロード
# 他のモデルを使いたい場合は URL と保存先ファイル名を変えてください（models/checkpoints/ 以下に .safetensors を置く）。
!wget -q --show-progress -O models/checkpoints/sd_v1-5.safetensors \
  https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors
print('ダウンロード完了')

models/checkpoints/ 100%[===================>]   1.99G  85.4MB/s    in 21s     
ダウンロード完了


In [3]:
# 3. cloudflared (トンネル用バイナリ) を取得
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('cloudflared 準備完了')

cloudflared 準備完了


In [ ]:
# 4. ComfyUI を起動し、cloudflared で外部公開 URL を発行
# --enable-cors-header は、別オリジン（ブラウザで開いている app/index.html）からの fetch を許可するために必須です。
import subprocess, threading, time, re

def stream_output(proc, prefix):
    for line in proc.stdout:
        print(prefix + line, end='')

comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--enable-cors-header'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
# ComfyUI自体のログをこのセルに流す（エラー時の原因調査に使う。生成が進まない/落ちた場合はここを確認）
threading.Thread(target=stream_output, args=(comfy_proc, '[ComfyUI] '), daemon=True).start()

# ComfyUI が起動するまで少し待つ
time.sleep(15)

tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

print('公開URLを取得中...')
url_found = False
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if m:
        print('
==================================')
        print(' ComfyUI 公開URL:', m.group(0))
        print(' → app/index.html の「バックエンドURL」欄に貼り付けてください')
        print('==================================
')
        url_found = True
        break
# トンネルのログも引き続きこのセルに流す（切断検知用）
threading.Thread(target=stream_output, args=(tunnel_proc, '[cloudflared] '), daemon=True).start()
print('ComfyUI と cloudflared はバックグラウンドで実行中です。このセルの実行は終了しても問題ありません。')
print('生成が止まる/進まない場合は、このセルの出力に [ComfyUI] のログが増えているか確認してください。')

## トラブルシューティング
- URLが表示されない場合: 上のセルをもう一度実行してください（トンネル接続に数秒かかることがあります）。
- アプリ側で「接続失敗」になる場合: URLの該写し(末尾の / など)を確認してください。
- 一定時間経つとColabセッションが切断されます。その場合は最初から全セルをやり直してください（URLが変わるのでアプリ側も更新）。